### Carlos Luengo
### CAP 5619
### November 2, 2026

# Assignment 2: SEC 8-K Product Announcement Extraction

### 1. Environment Setup

Install and import the required Python libraries, initialize Ollama with Llama 3.2, and connect Google Drive to store the results.

In [ ]:
!pip -q install requests pandas beautifulsoup4 ollama

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import os
import re
import json
import time
import subprocess
import requests
import pandas as pd

from bs4 import BeautifulSoup
from ollama import chat

# Start Ollama server
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Verify connection
response = requests.get("http://localhost:11434/api/version", timeout=10)
response.raise_for_status()

print("Ollama server is running.")

Ollama server is running.


In [ ]:
!ollama pull llama3.2

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Note: Update the directory to your own
drive_output_dir = "/content/drive"

if not os.path.isdir(drive_output_dir):
    raise FileNotFoundError("Google Drive project folder not found.")

print("Google Drive connected.")

Mounted at /content/drive
Google Drive connected.


### 2. SEC Data Collection

Use the SEC EDGAR API to retrieve company information and recent Form 8-K filings. Download the primary documents and relevant exhibits while respecting SEC request limits and handling connection errors.

In [ ]:
# SEC request identification
# Note: replace the email in the User-Agent with your own email
# (preferrably a school email)
headers = {
    "User-Agent": "Student SEC Research email@test.com",
    "Accept-Encoding": "gzip, deflate"
}

In [ ]:
REQUEST_DELAY = 0.5
MAX_RETRIES = 3

session = requests.Session()
session.headers.update(headers)

last_request_time = 0

def sec_request(url):
    global last_request_time

    for attempt in range(MAX_RETRIES):
        elapsed = time.monotonic() - last_request_time
        if elapsed < REQUEST_DELAY:
            time.sleep(REQUEST_DELAY - elapsed)

        try:
            last_request_time = time.monotonic()
            response = session.get(url, timeout=30)

            if response.status_code in [429, 500, 502, 503, 504]:
                if attempt == MAX_RETRIES - 1:
                    response.raise_for_status()

                try:
                    wait = float(response.headers.get("Retry-After"))
                except (TypeError, ValueError):
                    wait = 2 ** (attempt + 1)

                wait = min(max(wait, REQUEST_DELAY), 60)
                time.sleep(wait)
                continue

            response.raise_for_status()
            return response

        except requests.RequestException:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(2 ** (attempt + 1))

In [ ]:
def get_filing_exhibits(cik, accession_number):
    accession_clean = accession_number.replace("-", "")
    filing_directory = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{int(cik)}/{accession_clean}/"
    )

    index_url = filing_directory + f"{accession_number}-index.html"
    soup = BeautifulSoup(sec_request(index_url).content, "html.parser")

    exhibits = []

    for row in soup.select("table.tableFile tr"):
        cells = row.find_all("td")

        if len(cells) < 4:
            continue

        document_type = cells[3].get_text(" ", strip=True).upper()
        if not document_type.startswith("EX-99"):
            continue

        link = cells[2].find("a", href=True)
        if not link:
            continue

        filename = link["href"].split("/")[-1].split("?")[0]
        if not filename.lower().endswith((".htm", ".html")):
            continue

        exhibit_url = filing_directory + filename
        exhibit_text = clean_filing(sec_request(exhibit_url).content)

        exhibits.append({
            "filename": filename,
            "document_type": document_type,
            "url": exhibit_url,
            "text": exhibit_text
        })

    return exhibits

In [ ]:
ticker_url = "https://www.sec.gov/files/company_tickers.json"
ticker_data = pd.DataFrame.from_dict(
    sec_request(ticker_url).json(), orient="index"
)

ticker_data["cik_str"] = ticker_data["cik_str"].astype(str).str.zfill(10)
ticker_data["ticker"] = ticker_data["ticker"].str.upper().str.strip()

In [ ]:
def get_company_info(ticker):
    ticker = ticker.upper().strip()
    matches = ticker_data[ticker_data["ticker"] == ticker]

    if matches.empty:
        raise ValueError(f"Ticker '{ticker}' not found in SEC dataset.")

    company = matches.iloc[0]

    return {
        "company_name": company["title"],
        "stock_name": company["ticker"],
        "cik": company["cik_str"]
    }

In [ ]:
def get_company_8k_filings(ticker):
    company_info = get_company_info(ticker)
    cik = company_info["cik"]

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    filings = pd.DataFrame(sec_request(url).json()["filings"]["recent"])

    filings_8k = filings[filings["form"] == "8-K"].copy()

    filings_8k = filings_8k[
        ["form", "filingDate", "accessionNumber", "primaryDocument"]
    ].sort_values("filingDate", ascending=False).reset_index(drop=True)

    return company_info, filings_8k

### 3. Data Preprocessing

Clean the downloaded HTML documents using BeautifulSoup and remove unnecessary formatting. Filter the text to identify passages more likely to contain new product announcements, reducing the amount of unrelated information sent to the LLM.

In [ ]:
def clean_filing(html_content):
    soup = BeautifulSoup(html_content, "html.parser")

    for element in soup([
        "script", "style", "head", "noscript",
        "ix:header", "ix:hidden"
    ]):
        element.decompose()

    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()

In [ ]:
def find_product_passages(text, context=250):
    keywords = [
        r"\blaunch(?:ed|es|ing)?\b",
        r"\bintroduc(?:e|ed|es|ing|tion)\b",
        r"\bannounc(?:e|ed|es|ing|ement)\b",
        r"\bunveil(?:ed|s|ing)?\b",
        r"\breleas(?:e|ed|es|ing)\b",
        r"\bnew product(?:s)?\b",
        r"\bnew platform\b",
        r"\bnew device\b"
    ]

    pattern = re.compile("|".join(keywords), re.IGNORECASE)
    spans = [
        (max(0, m.start() - context), min(len(text), m.end() + context))
        for m in pattern.finditer(text)
    ]

    merged = []
    for start, end in spans:
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))

    return [text[start:end].strip() for start, end in merged]

### 4. LLM Product Extraction

Use Llama 3.2 to identify newly announced products from the filtered SEC documents. The prompt restricts extraction to the document's contents, while a JSON schema organizes product names, descriptions, and supporting evidence.

In [ ]:
product_schema = {
    "type": "object",
    "properties": {
        "products": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "new_product": {"type": "string"},
                    "product_description": {"type": "string", "maxLength": 179},
                    "evidence": {"type": "string"}
                },
                "required": ["new_product", "product_description", "evidence"],
                "additionalProperties": False
            }
        }
    },
    "required": ["products"],
    "additionalProperties": False
}

In [ ]:
def extract_products(text):
    prompt = f"""
Extract newly announced products from the following SEC Form 8-K document.

Rules:
1. Use ONLY the provided document. Do not use outside knowledge or invent information.
2. Include products explicitly announced, launched, introduced, or released.
3. Exclude existing products, previously announced products, and general product categories.
4. Copy each product's exact name from the document.
5. Describe the product using only information directly connected to it.
   Keep descriptions under 180 characters.
6. The evidence must be one continuous, verbatim passage copied
   from the provided document. Do not paraphrase, summarize,
   combine separate sentences, or change punctuation.
7. The evidence must contain the exact product name and clearly
   establish that the product was newly announced or launched.
   If no such passage exists, exclude the product.
8. If no products meet these requirements, return an empty products list.


Return the products in the required JSON structure.

Document:
{text}
"""


    response = chat(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        format=product_schema,
        options={
            "temperature": 0,
            "num_predict": 4096
        }
    )

    # Detect responses that stopped because of the output limit
    if response.get("done_reason") == "length":
        raise ValueError("LLM response exceeded the output token limit.")

    try:
        products = json.loads(response["message"]["content"]).get("products", [])
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid JSON returned by LLM: {error}") from error


    products = json.loads(response["message"]["content"]).get("products", [])

    if not isinstance(products, list):
        raise ValueError("Invalid product list returned by the LLM.")

    return products

### 5. Product Validation

Validate the extracted products by checking required fields, description lengths, duplicate entries, and supporting evidence against the original filing. Reject unsupported extractions and flag questionable results for review.

In [ ]:
def validate_products(products, source_text):
    valid, flagged, seen = [], [], set()
    source = re.sub(r"\s+", " ", source_text).strip()

    for product in products:
        if not isinstance(product, dict):
            flagged.append({"product": product, "reason": "Invalid product structure"})
            continue

        name = product.get("new_product", "")
        description = product.get("product_description", "")
        evidence = product.get("evidence", "")

        if not all(isinstance(x, str) for x in [name, description, evidence]):
            reason = "Invalid field type"
        else:
            name, description, evidence = name.strip(), description.strip(), evidence.strip()
            quote = re.sub(r"\s+", " ", evidence).strip()

            if not all([name, description, evidence]):
                reason = "Missing required information"
            elif len(description) >= 180:
                reason = "Description exceeds character limit"
            elif quote not in source:
                reason = "Evidence not found in source document"
            elif name.casefold() not in evidence.casefold():
                reason = "Product name missing from evidence"
            elif len(evidence) < 50:
                reason = "Evidence too short"
            elif name.casefold() in seen:
                reason = "Duplicate product"
            else:
                reason = None

        if reason:
            flagged.append({"product": product, "reason": reason})
            continue

        seen.add(name.casefold())
        valid.append({
            "new_product": name,
            "product_description": description,
            "evidence": evidence
        })

    return valid, flagged

In [ ]:
def extract_products_from_passages(text, chunk_size=12000):
    passages = find_product_passages(text)

    if not passages:
        return [], []

    valid_all, flagged_all = [], []
    current_chunk = ""

    # Split the filtered passages into smaller LLM requests
    chunks = []
    for passage in passages:
        if current_chunk and len(current_chunk) + len(passage) > chunk_size:
            chunks.append(current_chunk)
            current_chunk = ""

        if current_chunk:
            current_chunk += "\n\n"

        current_chunk += passage

    if current_chunk:
        chunks.append(current_chunk)

    # Extract and validate products from each chunk
    for chunk in chunks:
        products = extract_products(chunk)
        valid, flagged = validate_products(products, text)

        valid_all.extend(valid)
        flagged_all.extend(flagged)

    # Remove duplicate products across different chunks
    unique_products = []
    seen = set()

    for product in valid_all:
        name = product["new_product"].casefold()

        if name not in seen:
            seen.add(name)
            unique_products.append(product)

    return unique_products, flagged_all

### 6. Automated Batch Processing

Process multiple companies and their Form 8-K filings automatically. Save results to Google Drive and use checkpoints to skip previously completed filings, allowing interrupted batches to resume without repeating work.

In [ ]:
# Track successfully processed filings across Colab sessions
checkpoint_path = os.path.join(drive_output_dir, "processing_checkpoint.csv")
checkpoint_columns = ["stock_name", "accession_number", "filing_date", "status"]

def load_checkpoint():
    # A missing checkpoint indicates that no filings have been completed
    if not os.path.exists(checkpoint_path):
        return pd.DataFrame(columns=checkpoint_columns)

    checkpoint = pd.read_csv(checkpoint_path, dtype=str)

    # Prevent processing with an incorrectly formatted checkpoint
    if not set(checkpoint_columns).issubset(checkpoint.columns):
        raise ValueError("Invalid checkpoint file structure.")

    return checkpoint[checkpoint_columns]


def is_filing_completed(checkpoint, accession_number):
    # Only successfully completed filings are skipped
    return (
        (checkpoint["accession_number"] == accession_number)
        & (checkpoint["status"] == "completed")
    ).any()

In [ ]:
product_columns = [
    "company_name", "stock_name", "filing_time",
    "new_product", "product_description", "accession_number", "evidence"
]

flagged_columns = [
    "company_name", "stock_name", "filing_time",
    "accession_number", "product", "reason"
]

error_columns = ["stock_name", "accession_number", "filing_date", "error"]


def append_csv_records(filepath, records, columns, unique_columns):
    # Load previous results and remove duplicates before saving
    existing = (
        pd.read_csv(filepath, dtype=str)
        if os.path.exists(filepath)
        else pd.DataFrame(columns=columns)
    )

    combined = pd.concat(
        [existing, pd.DataFrame(records, columns=columns)],
        ignore_index=True
    ).drop_duplicates(subset=unique_columns, keep="last")

    # Write to a temporary file to reduce the risk of incomplete CSV files
    temporary_path = filepath + ".tmp"
    try:
        combined.to_csv(temporary_path, index=False)
        os.replace(temporary_path, filepath)
    finally:
        if os.path.exists(temporary_path):
            os.remove(temporary_path)

    return len(combined)


def save_filing_results(records, flagged, errors, stock_name,
                        accession_number, filing_date, completed=False):

    company_dir = os.path.join(drive_output_dir, stock_name)
    os.makedirs(company_dir, exist_ok=True)

    files = [
        ("valid_products.csv", records, product_columns,
         ["accession_number", "new_product"]),
        ("flagged_products.csv", flagged, flagged_columns,
         ["accession_number", "product", "reason"]),
        ("processing_errors.csv", errors, error_columns,
         ["accession_number", "error"])
    ]

    for filename, data, columns, unique in files:
        append_csv_records(
            os.path.join(company_dir, filename),
            data, columns, unique
        )

    if completed:
        append_csv_records(
            checkpoint_path,
            [{
                "stock_name": stock_name,
                "accession_number": accession_number,
                "filing_date": filing_date,
                "status": "completed"
            }],
            checkpoint_columns,
            ["accession_number"]
        )

In [ ]:
def process_filing(filing, cik, company_name, stock_name):
    accession = filing["accessionNumber"]
    filing_date = filing["filingDate"]
    accession_clean = accession.replace("-", "")

    directory = (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{int(cik)}/{accession_clean}/"
    )

    # Download and clean the primary filing and its EX-99 exhibits
    primary_text = clean_filing(
        sec_request(directory + filing["primaryDocument"]).content
    )

    exhibits = get_filing_exhibits(cik, accession)

    # Combine all documents before extracting product announcements
    combined_text = "\n\n".join(
        [primary_text] + [exhibit["text"] for exhibit in exhibits]
    )

    # Extract potential new products and validate their evidence
    valid, flagged = extract_products_from_passages(combined_text)

    # Prepare valid records, preserving evidence for later review
    records = [{
        "company_name": company_name,
        "stock_name": stock_name,
        "filing_time": filing_date,
        "new_product": product["new_product"],
        "product_description": product["product_description"],
        "accession_number": accession,
        "evidence": product["evidence"]
    } for product in valid]

    # Store rejected extractions separately with their reasons
    flagged_records = [{
        "company_name": company_name,
        "stock_name": stock_name,
        "filing_time": filing_date,
        "accession_number": accession,
        "product": str(item["product"]),
        "reason": item["reason"]
    } for item in flagged]

    print(
        f"{accession}: {len(records)} valid, "
        f"{len(flagged_records)} flagged"
    )

    return records, flagged_records

In [ ]:
def process_batch(filings, cik, company_name, stock_name, limit=10):
    records_all, flagged_all, errors_all = [], [], []

    # Load previously completed filings to avoid repeating work
    checkpoint = load_checkpoint()
    completed = skipped = 0

    for _, filing in filings.head(limit).iterrows():
        accession = filing["accessionNumber"]
        filing_date = filing["filingDate"]

        # Skip filings that already have a successful checkpoint
        if is_filing_completed(checkpoint, accession):
            skipped += 1
            continue

        try:
            # Download, preprocess, extract, and validate
            records, flagged = process_filing(
                filing, cik, company_name, stock_name
            )

            # Save the results and mark the filing as completed
            save_filing_results(
                records, flagged, [], stock_name,
                accession, filing_date, completed=True
            )

            records_all.extend(records)
            flagged_all.extend(flagged)
            completed += 1
            checkpoint = load_checkpoint()

        except Exception as error:
            # Record failed filings so they can be retried later
            error_record = {
                "stock_name": stock_name,
                "accession_number": accession,
                "filing_date": filing_date,
                "error": str(error)
            }

            errors_all.append(error_record)
            print(f"Error processing {accession}: {error}")

            # Save the error without marking the filing as completed
            save_filing_results(
                [], [], [error_record], stock_name,
                accession, filing_date, completed=False
            )

    print(f"{stock_name}: {completed} completed, {skipped} skipped, "
          f"{len(records_all)} products, {len(errors_all)} errors")

    return records_all, flagged_all, errors_all

In [ ]:
def process_multiple_companies(tickers, filings_per_company=10):
    all_records, all_flagged, all_errors = [], [], []

    for ticker in tickers:
        print(f"\nProcessing company: {ticker}")

        try:
            # Retrieve the company's metadata and Form 8-K filings
            company, filings = get_company_8k_filings(ticker)

            # Process the filings using the checkpoint-enabled function
            records, flagged, errors = process_batch(
                filings=filings,
                cik=company["cik"],
                company_name=company["company_name"],
                stock_name=company["stock_name"],
                limit=filings_per_company
            )

            # Combine results from all companies
            all_records.extend(records)
            all_flagged.extend(flagged)
            all_errors.extend(errors)

        except Exception as error:
            print(f"Error processing {ticker}: {error}")
            all_errors.append({"ticker": ticker, "error": str(error)})

    print("\nMulti-company processing complete.")
    print("New products:", len(all_records))
    print("Flagged products:", len(all_flagged))
    print("Errors:", len(all_errors))

    return all_records, all_flagged, all_errors

In [ ]:
company_tickers = [
    "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL",
    "META", "AMD", "INTC", "ORCL", "CRM"
]

batch_records, batch_flagged, batch_errors = process_multiple_companies(
    company_tickers,
    filings_per_company=10
)


Processing company: AAPL
0000320193-26-000018: 2 valid, 0 flagged
0000320193-26-000011: 1 valid, 2 flagged
0001140361-26-015711: 0 valid, 0 flagged
0001140361-26-006577: 0 valid, 0 flagged
0000320193-26-000005: 1 valid, 1 flagged
0001140361-26-000199: 0 valid, 0 flagged
0001140361-25-044561: 0 valid, 0 flagged
0000320193-25-000077: 5 valid, 3 flagged
0000320193-25-000071: 0 valid, 1 flagged
0001140361-25-027340: 0 valid, 0 flagged
AAPL: 10 completed, 0 skipped, 9 products, 0 errors

Processing company: MSFT
0001193125-26-380280: 0 valid, 0 flagged
0001193125-26-323632: 1 valid, 1 flagged
0001193125-26-258667: 0 valid, 0 flagged
0001193125-26-224155: 0 valid, 1 flagged
0001193125-26-191457: 0 valid, 2 flagged
0001193125-26-027198: 1 valid, 1 flagged
0001193125-25-311196: 0 valid, 0 flagged
0001193125-25-256310: 1 valid, 1 flagged
0001193125-25-225125: 0 valid, 0 flagged
0000950170-25-100226: 0 valid, 2 flagged
MSFT: 10 completed, 0 skipped, 3 products, 0 errors

Processing company: NVD

### 7. Results and CSV Export

Combine the extracted products, apply manual review decisions, and export the accepted products into the required five-column CSV. Display a summary of the final results.

In [ ]:
all_products = []

for ticker in company_tickers:
    path = os.path.join(drive_output_dir, ticker, "valid_products.csv")

    if os.path.exists(path):
        all_products.append(pd.read_csv(path))

products_df = (
    pd.concat(all_products, ignore_index=True)
    .drop_duplicates(["accession_number", "new_product"])
    if all_products
    else pd.DataFrame(columns=product_columns)
)

print("Total product candidates:", len(products_df))

if not products_df.empty:
    display(
        products_df.groupby("stock_name")
        .size()
        .reset_index(name="products_found")
    )

Total product candidates: 79


,stock_name,products_found
0,AAPL,9
1,AMD,15
2,AMZN,19
3,CRM,6
4,GOOGL,1
5,INTC,1
6,META,2
7,MSFT,3
8,NVDA,14
9,ORCL,9


#### Manual Review

In [ ]:
# Display Apple's candidates with descriptions and evidence
apple_review = review_df[
    review_df["stock_name"] == "AAPL"
]

for _, product in apple_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])
    print()

Review ID: 1
Product: AirPods Pro 3
Filing date: 2025-10-30
Accession: 0000320193-25-000077
Description: Fantastic AirPods Pro 3
Evidence: In addition, we launched the fantastic AirPods Pro 3

Review ID: 2
Product: MacBook Pro
Filing date: 2025-10-30
Accession: 0000320193-25-000077
Description: Recently announced MacBook Pro
Evidence: When combined with the recently announced MacBook Pro

Review ID: 3
Product: iPhone 17
Filing date: 2025-10-30
Accession: 0000320193-25-000077
Description: Best iPhone lineup ever, including iPhone 17, iPhone 17 Pro and Pro Max, and iPhone Air
Evidence: In September, we were thrilled to launch our best iPhone lineup ever, including iPhone 17, iPhone 17 Pro and Pro Max, and iPhone Air.

Review ID: 4
Product: iPhone 17 Pro
Filing date: 2025-10-30
Accession: 0000320193-25-000077
Description: Best iPhone lineup ever, including iPhone 17, iPhone 17 Pro and Pro Max, and iPhone Air
Evidence: In September, we were thrilled to launch our best iPhone lineup ever, i

In [ ]:
# Record Apple's manual review decisions
apple_decisions = {
    1: ("accepted", "Explicit product launch",
        "Apple announced the launch of AirPods Pro 3."),
    2: ("accepted", "Recently announced product",
        "Apple referenced a recently announced MacBook Pro."),
    3: ("accepted", "Explicit product launch",
        "Apple launched iPhone 17 in September as part of its new iPhone lineup."),
    4: ("accepted", "Explicit product launch",
        "Apple launched iPhone 17 Pro in September as part of its new iPhone lineup."),
    5: ("accepted", "Explicit product launch",
        "Apple launched iPhone Air in September as part of its new iPhone lineup."),
    6: ("rejected", "Existing product; sales performance only", None),
    7: ("accepted", "Explicit product launch",
        "Apple launched MacBook Neo, describing strong customer interest worldwide."),
    8: ("rejected", "Existing product mentioned without a new launch", None),
    9: ("accepted", "Explicit new product introduction",
        "Apple introduced the all-new Siri AI at WWDC26 alongside its latest software innovations.")
}

# Update the existing review file
review_df = pd.read_csv(review_path)

for review_id, (status, reason, description) in apple_decisions.items():
    mask = review_df["review_id"] == review_id
    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

    if description is not None:
        review_df.loc[mask, "product_description"] = description

review_df.to_csv(review_path, index=False)

print("Apple review completed.")
display(review_df.loc[
    review_df["stock_name"] == "AAPL",
    ["review_id", "new_product", "review_status", "review_reason"]
])

Apple review completed.


/tmp/ipykernel_456/3656019533.py:27: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Explicit product launch' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  review_df.loc[mask, "review_reason"] = reason


,review_id,new_product,review_status,review_reason
0,1,AirPods Pro 3,accepted,Explicit product launch
1,2,MacBook Pro,accepted,Recently announced product
2,3,iPhone 17,accepted,Explicit product launch
3,4,iPhone 17 Pro,accepted,Explicit product launch
4,5,iPhone Air,accepted,Explicit product launch
5,6,iPhone,rejected,Existing product; sales performance only
6,7,MacBook Neo,accepted,Explicit product launch
7,8,Apple Vision Pro,rejected,Existing product mentioned without a new launch
8,9,Siri AI,accepted,Explicit new product introduction


In [ ]:
# Display Microsoft's candidates for manual review
microsoft_review = review_df[
    review_df["stock_name"] == "MSFT"
]

for _, product in microsoft_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])
    print()

Review ID: 54
Product: Microsoft Cloud and AI Strength Drives First Quarter Results
Filing date: 2025-10-29
Accession: 0001193125-25-256310
Description: Press release announcing financial results for the fiscal quarter ended September 30, 2025
Evidence: Microsoft Cloud and AI Strength Drives First Quarter Results REDMOND, Wash. — October 29, 2025 — Microsoft Corp. today announced the following results for the quarter ended September 30, 2025, as compared to the corresponding period of last fiscal year:

Review ID: 55
Product: Microsoft Cloud
Filing date: 2026-01-28
Accession: 0001193125-26-027198
Description: Cloud computing service offered by Microsoft Corporation
Evidence: Microsoft Cloud and AI Strength Drives Second Quarter Results

Review ID: 56
Product: OpenAI
Filing date: 2026-07-29
Accession: 0001193125-26-323632
Description: Business acquired by Microsoft, with impact on financial results and business outlook, as announced on July 29, 2026, as part of its financial results for

In [ ]:
# Record Microsoft's manual review decisions
microsoft_decisions = {
    54: ("rejected", "Financial results press release, not a product"),
    55: ("rejected", "Existing cloud services without a new launch"),
    56: ("rejected", "Investment-related financial reference, not a product")
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason) in microsoft_decisions.items():
    mask = review_df["review_id"] == review_id
    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

review_df.to_csv(review_path, index=False)

print("Microsoft review completed.")

display(review_df.loc[
    review_df["stock_name"] == "MSFT",
    ["review_id", "new_product", "review_status", "review_reason"]
])

Microsoft review completed.


,review_id,new_product,review_status,review_reason
53,54,Microsoft Cloud and AI Strength Drives First Q...,rejected,"Financial results press release, not a product"
54,55,Microsoft Cloud,rejected,Existing cloud services without a new launch
55,56,OpenAI,rejected,"Investment-related financial reference, not a ..."


In [ ]:
# Display NVIDIA's candidates for manual review
nvidia_review = review_df[
    review_df["stock_name"] == "NVDA"
]

for _, product in nvidia_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])
    print()

Review ID: 57
Product: NVIDIA DLSS 4.5 Dynamic Multi Frame Generation
Filing date: 2026-05-20
Accession: 0001045810-26-000051
Description: previewed the next generation of DLSS 3D-guided neural rendering model, DLSS 5
Evidence: Released NVIDIA DLSS 4.5 Dynamic Multi Frame Generation and previewed the next generation of DLSS 3D-guided neural rendering model, DLSS 5, NVIDIA’s most significant graphics breakthrough since ray tracing in 2018.

Review ID: 58
Product: NVIDIA Dynamo 1.0
Filing date: 2026-05-20
Accession: 0001045810-26-000051
Description: open source software that boosts generative and agentic inference on NVIDIA Blackwell GPUs by up to 7x
Evidence: Entered production with NVIDIA Dynamo 1.0, open source software that boosts generative and agentic inference on NVIDIA Blackwell GPUs by up to 7x, with widespread global adoption.

Review ID: 59
Product: NVIDIA NVLink Fusion
Filing date: 2026-05-20
Accession: 0001045810-26-000051
Description: silicon photonics technology
Evidence: 

In [ ]:
# Record NVIDIA's manual review decisions
nvidia_decisions = {
    57: ("accepted", "Explicit product release",
         "NVIDIA released DLSS 4.5 Dynamic Multi Frame Generation alongside a preview of its upcoming DLSS 5 technology."),
    58: ("accepted", "New software version entered production",
         "Open-source software for generative and agentic AI inference, delivering up to 7x improvement on NVIDIA Blackwell GPUs."),
    59: ("rejected", "Existing technology referenced in a partnership", None),
    60: ("rejected", "Community funding initiative, not a product", None),
    61: ("accepted", "Explicit introduction of a new AI model",
         "Open reasoning model introduced for commercial robotaxi and autonomous vehicle development."),
    62: ("rejected", "Existing hardware discussed in performance benchmarks", None),
    63: ("accepted", "New Windows offering explicitly announced",
         "NVIDIA announced DGX Station for Windows, a deskside AI supercomputer for developing and running AI agents."),
    64: ("rejected", "Expansion of an existing platform ecosystem", None),
    65: ("accepted", "Explicit platform launch",
         "Platform launched to help infrastructure builders design, build, and operate large-scale AI factories."),
    66: ("accepted", "Product announced as entering full production",
         "Interactive AI inference accelerator announced as entering full production."),
    67: ("accepted", "Explicit product announcement",
         "Full-stack physical AI safety system combining AI computing and safety capabilities in one platform."),
    68: ("rejected", "Existing AI models referenced in collaborations", None),
    69: ("accepted", "Explicit unveiling of a new CPU",
         "New CPU designed for AI agents, with adoption planned across major technology providers."),
    70: ("rejected", "Existing platform ramping production without a clear new launch", None)
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason, description) in nvidia_decisions.items():
    mask = (review_df["review_id"] == review_id) & (review_df["stock_name"] == "NVDA")
    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

    if description is not None:
        review_df.loc[mask, "product_description"] = description

review_df.to_csv(review_path, index=False)

print("NVIDIA review completed.")
display(review_df.loc[
    review_df["stock_name"] == "NVDA",
    ["review_id", "new_product", "review_status", "review_reason"]
])

NVIDIA review completed.


,review_id,new_product,review_status,review_reason
56,57,NVIDIA DLSS 4.5 Dynamic Multi Frame Generation,accepted,Explicit product release
57,58,NVIDIA Dynamo 1.0,accepted,New software version entered production
58,59,NVIDIA NVLink Fusion,rejected,Existing technology referenced in a partnership
59,60,Community benefits fund,rejected,"Community funding initiative, not a product"
60,61,NVIDIA Alpamayo 2 Super,accepted,Explicit introduction of a new AI model
61,62,NVIDIA Blackwell,rejected,Existing hardware discussed in performance ben...
62,63,NVIDIA DGX Station,accepted,New Windows offering explicitly announced
63,64,NVIDIA DRIVE Hyperion,rejected,Expansion of an existing platform ecosystem
64,65,NVIDIA DSX,accepted,Explicit platform launch
65,66,NVIDIA Groq 3 LPX,accepted,Product announced as entering full production


In [ ]:
# Display Amazon's candidates for manual review
amazon_review = review_df[
    review_df["stock_name"] == "AMZN"
]

for _, product in amazon_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])
    print()

Review ID: 25
Product: Prime Air
Filing date: 2026-04-09
Accession: 0001104659-26-041034
Description: Delivery service using drones with SSDs for gathering selection and launching, serving different needs and driving inflection points in the grocery market
Evidence: The drones will use SSDs as a building block from which to gather selection and launch. And, while Amazon Now will deliver thousands of items within 20 minutes by leveraging micro-fulfillment centers, Prime Air will deliver a much larger selection of items inside a half hour.

Review ID: 26
Product: Amazon Leo
Filing date: 2026-04-14
Accession: 0001104659-26-042880
Description: Direct-to-device (D2D) services for low Earth orbit satellite network and extended cellular coverage beyond terrestrial networks
Evidence: enabling Amazon Leo to add direct-to-device (D2D) services to its low Earth orbit satellite network and extend cellular coverage to customers beyond the reach of terrestrial networks.

Review ID: 27
Product: Amazo

In [ ]:
# Record Amazon's manual review decisions
amazon_decisions = {
    25: ("rejected", "Existing drone delivery service", None),
    26: ("rejected", "Future capabilities for an existing service", None),
    27: ("accepted", "New advertising offering for Netflix",
         "Advertising offering using Amazon shopping, streaming, and browsing signals to help brands reach relevant audiences on Netflix."),
    28: ("rejected", "New capabilities for an existing service", None),
    29: ("accepted", "Explicit announcement of a product preview",
         "Preview of an OpenAI-powered runtime environment for building generative AI applications and agents at production scale."),
    30: ("accepted", "Explicit product launch",
         "Agentic AI application for drug discovery, providing biological foundation models, experimental design agents, and laboratory partners."),
    31: ("rejected", "Future capabilities following a planned acquisition", None),
    32: ("rejected", "Future airline service agreement, not a product launch", None),
    33: ("rejected", "Existing service expanding into additional markets", None),
    34: ("rejected", "Third-party model available through Amazon Bedrock", None),
    35: ("rejected", "Third-party model available through Amazon Bedrock", None),
    36: ("accepted", "Explicit launch of an advertising solution",
         "AI advertising solution that helps brands research, brainstorm, and generate advertising campaigns."),
    37: ("rejected", "Existing robotaxi service and partnership", None),
    38: ("rejected", "Additional capabilities for an existing product", None),
    39: ("accepted", "New general availability announcement",
         "Cloud platform made generally available to help defense contractors run classified workloads securely."),
    40: ("rejected", "Existing product expanding to more countries", None),
    41: ("rejected", "Satellite deployment, not a new product launch", None),
    42: ("accepted", "Explicit service launch",
         "Service enabling businesses to move, store, and deliver goods using Amazon's supply chain infrastructure."),
    43: ("accepted", "Explicit introduction of a next-generation robot",
         "Next-generation autonomous fulfillment robot that moves goods weighing up to 1,300 pounds and supports employee safety.")
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason, description) in amazon_decisions.items():
    mask = (
        (review_df["review_id"] == review_id) &
        (review_df["stock_name"] == "AMZN")
    )

    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

    if description is not None:
        review_df.loc[mask, "product_description"] = description

review_df.to_csv(review_path, index=False)

print("Amazon review completed.")

display(review_df.loc[
    review_df["stock_name"] == "AMZN",
    ["review_id", "new_product", "review_status", "review_reason"]
])

Amazon review completed.


,review_id,new_product,review_status,review_reason
24,25,Prime Air,rejected,Existing drone delivery service
25,26,Amazon Leo,rejected,Future capabilities for an existing service
26,27,Amazon Audiences,accepted,New advertising offering for Netflix
27,28,Amazon Bedrock AgentCore,rejected,New capabilities for an existing service
28,29,Amazon Bedrock Managed Agents,accepted,Explicit announcement of a product preview
29,30,Amazon Bio Discovery,accepted,Explicit product launch
30,31,Amazon Leo,rejected,Future capabilities following a planned acquis...
31,32,Amazon Leo-powered Wi-Fi,rejected,"Future airline service agreement, not a produc..."
32,33,Amazon Now,rejected,Existing service expanding into additional mar...
33,34,Claude Mythos Preview,rejected,Third-party model available through Amazon Bed...


In [ ]:
# Display Google's candidate for manual review
google_review = review_df[review_df["stock_name"] == "GOOGL"]

for _, product in google_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])

Review ID: 50
Product: Alphabet Announces First Quarter 2026 Results
Filing date: 2026-04-29
Accession: 0001652044-26-000043
Description: Financial results for the quarter ended March 31, 2026
Evidence: Alphabet Announces First Quarter 2026 Results MOUNTAIN VIEW, Calif. – April 29, 2026 – Alphabet Inc. (NASDAQ: GOOG, GOOGL) today announced financial results for the quarter ended March 31, 2026.


In [ ]:
# Record Google's manual review decision
google_decisions = {
    50: ("rejected", "Financial results announcement, not a product")
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason) in google_decisions.items():
    mask = (
        (review_df["review_id"] == review_id) &
        (review_df["stock_name"] == "GOOGL")
    )

    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

review_df.to_csv(review_path, index=False)

print("Google review completed.")

display(review_df.loc[
    review_df["stock_name"] == "GOOGL",
    ["review_id", "new_product", "review_status", "review_reason"]
])

Google review completed.


,review_id,new_product,review_status,review_reason
49,50,Alphabet Announces First Quarter 2026 Results,rejected,"Financial results announcement, not a product"


In [ ]:
# Display Meta's candidates for manual review
meta_review = review_df[review_df["stock_name"] == "META"]

for _, product in meta_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])
    print()

Review ID: 52
Product: Muse
Filing date: 2026-10-21
Accession: 0001628280-26-066700
Description: Our first frontier model
Evidence: We hit major milestones this quarter toward delivering personal superintelligence for everyone - including launching Muse and Muse Wonder, our first frontier model,

Review ID: 53
Product: Muse Wonder
Filing date: 2026-10-21
Accession: 0001628280-26-066700
Description: Our first frontier model
Evidence: We hit major milestones this quarter toward delivering personal superintelligence for everyone - including launching Muse and Muse Wonder, our first frontier model,



In [ ]:
# Record Meta's manual review decisions
meta_decisions = {
    52: (
        "accepted",
        "Explicit product launch",
        "Meta launched Muse as part of its efforts to develop personal superintelligence."
    ),
    53: (
        "accepted",
        "Explicit launch of a frontier AI model",
        "Meta launched Muse Wonder, described as its first frontier AI model, as part of its personal superintelligence initiative."
    )
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason, description) in meta_decisions.items():
    mask = (
        (review_df["review_id"] == review_id) &
        (review_df["stock_name"] == "META")
    )

    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason
    review_df.loc[mask, "product_description"] = description

review_df.to_csv(review_path, index=False)

print("Meta review completed.")

display(review_df.loc[
    review_df["stock_name"] == "META",
    ["review_id", "new_product", "review_status", "review_reason"]
])

Meta review completed.


,review_id,new_product,review_status,review_reason
51,52,Muse,accepted,Explicit product launch
52,53,Muse Wonder,accepted,Explicit launch of a frontier AI model


In [ ]:
# Display AMD's candidates for manual review
amd_review = review_df[review_df["stock_name"] == "AMD"]

for _, product in amd_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])
    print()

Review ID: 10
Product: AMD Helios rack-scale platform
Filing date: 2026-02-03
Accession: 0000002488-26-000014
Description: yotta-scale AI infrastructure
Evidence: ◦ An early look at the AMD Helios rack-scale platform, the blueprint for yotta-scale AI infrastructure.

Review ID: 11
Product: AMD Instinct MI440X GPU
Filing date: 2026-02-03
Accession: 0000002488-26-000014
Description: enterprise AI GPU
Evidence: ◦ Announcing the AMD Instinct MI440X GPU for enterprise AI.

Review ID: 12
Product: AMD Ryzen AI 400 and PRO 400 Series platforms
Filing date: 2026-02-03
Accession: 0000002488-26-000014
Description: AI-optimized CPU platforms
Evidence: ◦ New AMD Ryzen AI 400 and PRO 400 Series platforms, Ryzen AI Max+ SKUs and the AMD Ryzen AI Halo developer platform.

Review ID: 13
Product: AMD Ryzen AI Embedded Processor portfolio
Filing date: 2026-02-03
Accession: 0000002488-26-000014
Description: AI-driven applications across automotive, industrial automation and physical AI
Evidence: ◦ New AMD

In [ ]:
# Record AMD's manual review decisions
amd_decisions = {
    10: ("rejected", "Early preview without a clear product launch", None),
    11: ("accepted", "Explicit GPU announcement",
         "GPU announced for enterprise AI applications."),
    12: ("accepted", "New computing platforms announced",
         "New Ryzen AI 400 and PRO 400 Series computing platforms announced by AMD."),
    13: ("accepted", "New processor portfolio announced",
         "New processor portfolio designed for AI applications in automotive, industrial automation, and physical AI."),
    14: ("rejected", "Existing processors used in Cisco equipment", None),
    15: ("accepted", "Explicit FPGA family launch",
         "New mid-range FPGA family offering advanced memory bandwidth and I/O performance for industrial, imaging, and broadcast applications."),
    16: ("accepted", "Explicit new processor announcement",
         "New processors providing scalable, power-efficient AI computing for industrial and edge applications."),
    17: ("accepted", "Explicit launch of rack-scale solution",
         "Rack-scale AI infrastructure solution launched with AMD's broader data-center portfolio."),
    18: ("accepted", "Explicit GPU launch",
         "GPUs launched to bring AI acceleration to existing computing infrastructure."),
    19: ("rejected", "Strategic deployment agreement, not a new product launch", None),
    20: ("accepted", "Explicit announcement of physical AI solutions",
         "Physical AI solutions including AMD Kria AI SOMs and the AMD Kria AI Robotics Developer Platform."),
    21: ("accepted", "Explicit introduction of new processors",
         "Processors combining x86 CPU, GPU, and AI acceleration for robotics, industrial automation, and embedded systems."),
    22: ("accepted", "Newly expanded processor offering",
         "Expanded processor lineup bringing AMD 3D V-Cache technology to enterprise workstations for the first time."),
    23: ("accepted", "Explicit product release",
         "New systems released for local AI computing with AMD ROCm open software."),
    24: ("accepted", "Explicit adaptive SoC launch",
         "Adaptive SoCs designed to accelerate data transfers for measurement, video, communications, aerospace, and defense workloads.")
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason, description) in amd_decisions.items():
    mask = (
        (review_df["review_id"] == review_id) &
        (review_df["stock_name"] == "AMD")
    )

    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

    if description is not None:
        review_df.loc[mask, "product_description"] = description

review_df.to_csv(review_path, index=False)

print("AMD review completed.")

display(review_df.loc[
    review_df["stock_name"] == "AMD",
    ["review_id", "new_product", "review_status", "review_reason"]
])

AMD review completed.


,review_id,new_product,review_status,review_reason
9,10,AMD Helios rack-scale platform,rejected,Early preview without a clear product launch
10,11,AMD Instinct MI440X GPU,accepted,Explicit GPU announcement
11,12,AMD Ryzen AI 400 and PRO 400 Series platforms,accepted,New computing platforms announced
12,13,AMD Ryzen AI Embedded Processor portfolio,accepted,New processor portfolio announced
13,14,AMD Ryzen Embedded V3000 processors,rejected,Existing processors used in Cisco equipment
14,15,Kintex UltraScale+ Gen 2 family of mid-range F...,accepted,Explicit FPGA family launch
15,16,Ryzen AI Embedded P100 Series processors,accepted,Explicit new processor announcement
16,17,AMD Helios,accepted,Explicit launch of rack-scale solution
17,18,AMD Instinct MI350P GPUs,accepted,Explicit GPU launch
18,19,AMD Instinct MI450 Series GPUs,rejected,"Strategic deployment agreement, not a new prod..."


In [ ]:
# Display Intel's candidate for manual review
intel_review = review_df[review_df["stock_name"] == "INTC"]

for _, product in intel_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])

Review ID: 51
Product: Intel Xeon 6
Filing date: 2026-04-23
Accession: 0000050863-26-000077
Description: Host CPU for NVIDIA’s DGX Rubin NVL8 systems
Evidence: Intel Xeon 6 was selected as the host CPU for NVIDIA’s DGX Rubin NVL8 systems, reinforcing Intel’s continued role at the center of leading AI infrastructure deployments.


In [ ]:
# Record Intel's manual review decision
intel_decisions = {
    51: ("rejected", "Existing processor selected for NVIDIA systems, not a new product launch")
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason) in intel_decisions.items():
    mask = (
        (review_df["review_id"] == review_id) &
        (review_df["stock_name"] == "INTC")
    )

    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

review_df.to_csv(review_path, index=False)

print("Intel review completed.")

display(review_df.loc[
    review_df["stock_name"] == "INTC",
    ["review_id", "new_product", "review_status", "review_reason"]
])

Intel review completed.


,review_id,new_product,review_status,review_reason
50,51,Intel Xeon 6,rejected,Existing processor selected for NVIDIA systems...


In [ ]:
# Display Oracle's candidates for manual review
oracle_review = review_df[review_df["stock_name"] == "ORCL"]

for _, product in oracle_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])
    print()

Review ID: 71
Product: NetSuite Cloud ERP (SaaS) Revenue
Filing date: 2025-12-10
Accession: 0001193125-25-314207
Description: $1.0 billion, up 13% in both USD and constant currency
Evidence: • Q2 NetSuite Cloud ERP (SaaS) Revenue $1.0 billion, up 13% in both USD and constant currency

Review ID: 72
Product: Q2 Remaining Performance Obligations
Filing date: 2025-12-10
Accession: 0001193125-25-314207
Description: $523 billion, up 438% in USD
Evidence: • Q2 Remaining Performance Obligations $523 billion, up 438% in USD

Review ID: 73
Product: Hilary Maxson
Filing date: 2026-04-06
Accession: 0001193125-26-142939
Description: Chief Financial Officer (CFO) appointment
Evidence: Oracle Appoints Hilary Maxson as Chief Financial Officer

Review ID: 74
Product: Oracle Names Tomislav Mihaljevic, M.D., to the Board of Directors
Filing date: 2026-05-12
Accession: 0001193125-26-219708
Description: Press Release announcing the appointment of Dr. Mihaljevic to the Board of Directors
Evidence: Oracle N

In [ ]:
# Record Oracle's manual review decisions
oracle_decisions = {
    71: ("rejected", "Revenue metric for an existing product"),
    72: ("rejected", "Financial performance metric"),
    73: ("rejected", "Executive appointment, not a product"),
    74: ("rejected", "Board appointment, not a product"),
    75: ("rejected", "Cloud services revenue metric"),
    76: ("rejected", "Cloud infrastructure revenue metric"),
    77: ("rejected", "Earnings-per-share financial metric"),
    78: ("rejected", "Financial performance metric"),
    79: ("rejected", "Stock-sale plan cancellation, not a product")
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason) in oracle_decisions.items():
    mask = (
        (review_df["review_id"] == review_id) &
        (review_df["stock_name"] == "ORCL")
    )

    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

review_df.to_csv(review_path, index=False)

print("Oracle review completed.")

display(review_df.loc[
    review_df["stock_name"] == "ORCL",
    ["review_id", "new_product", "review_status", "review_reason"]
])

Oracle review completed.


,review_id,new_product,review_status,review_reason
70,71,NetSuite Cloud ERP (SaaS) Revenue,rejected,Revenue metric for an existing product
71,72,Q2 Remaining Performance Obligations,rejected,Financial performance metric
72,73,Hilary Maxson,rejected,"Executive appointment, not a product"
73,74,"Oracle Names Tomislav Mihaljevic, M.D., to the...",rejected,"Board appointment, not a product"
74,75,Cloud Apps (SaaS) Revenue,rejected,Cloud services revenue metric
75,76,Cloud Infra (IaaS) Revenue,rejected,Cloud infrastructure revenue metric
76,77,Record Q4 Earnings per Share GAAP,rejected,Earnings-per-share financial metric
77,78,Record Remaining Performance Obligations,rejected,Financial performance metric
78,79,Larry Ellison Cancels His Plan to Sell Oracle ...,rejected,"Stock-sale plan cancellation, not a product"


In [ ]:
# Display Salesforce's candidates for manual review
salesforce_review = review_df[review_df["stock_name"] == "CRM"]

for _, product in salesforce_review.iterrows():
    print("=" * 70)
    print("Review ID:", product["review_id"])
    print("Product:", product["new_product"])
    print("Filing date:", product["filing_time"])
    print("Accession:", product["accession_number"])
    print("Description:", product["product_description"])
    print("Evidence:", product["evidence"])
    print()

Review ID: 44
Product: Accelerated Share Repurchase
Filing date: 2026-03-16
Accession: 0001193125-26-107403
Description: prepayment and initial share delivery under accelerated share repurchase agreements
Evidence: Salesforce (NYSE: CRM), the world’s #1 AI CRM, today commenced the prepayment and initial delivery of approximately 103 million shares under its previously announced $25 billion accelerated share repurchase (“ASR”) agreements

Review ID: 45
Product: Public Sector Industry Cloud
Filing date: 2026-05-27
Accession: 0001108524-26-000125
Description: Public Sector Industry Cloud ARR surpasses $2 billion, up 23% Y/Y
Evidence: Public Sector Industry Cloud ARR surpasses $2 billion, up 23% Y/Y

Review ID: 46
Product: Salesforce Delivers Record First Quarter Fiscal 2027 Results
Filing date: 2026-05-27
Accession: 0001108524-26-000125
Description: GAAP EPS $2.42, up 52% Y/Y, Non-GAAP EPS $3.88, up 50% Y/Y
Evidence: Salesforce Delivers Record First Quarter Fiscal 2027 Results GAAP EPS $2

In [ ]:
# Record Salesforce's manual review decisions
salesforce_decisions = {
    44: ("rejected", "Stock repurchase transaction, not a product"),
    45: ("rejected", "Existing cloud service discussed in revenue results"),
    46: ("rejected", "Financial earnings announcement, not a product"),
    47: ("rejected", "Existing platform referenced in ARR reporting"),
    48: ("rejected", "Financial earnings announcement, not a product"),
    49: ("rejected", "Existing product capability discussed through usage metrics")
}

review_df = pd.read_csv(review_path)

for review_id, (status, reason) in salesforce_decisions.items():
    mask = (
        (review_df["review_id"] == review_id) &
        (review_df["stock_name"] == "CRM")
    )

    review_df.loc[mask, "review_status"] = status
    review_df.loc[mask, "review_reason"] = reason

review_df.to_csv(review_path, index=False)

print("Salesforce review completed.")

display(review_df.loc[
    review_df["stock_name"] == "CRM",
    ["review_id", "new_product", "review_status", "review_reason"]
])

Salesforce review completed.


,review_id,new_product,review_status,review_reason
43,44,Accelerated Share Repurchase,rejected,"Stock repurchase transaction, not a product"
44,45,Public Sector Industry Cloud,rejected,Existing cloud service discussed in revenue re...
45,46,Salesforce Delivers Record First Quarter Fisca...,rejected,"Financial earnings announcement, not a product"
46,47,Agentforce,rejected,Existing platform referenced in ARR reporting
47,48,Salesforce Delivers Record Second Quarter Fisc...,rejected,"Financial earnings announcement, not a product"
48,49,Zero Copy,rejected,Existing product capability discussed through ...


#### Final CSV File

In [ ]:
# Verify that all manual reviews have been completed
review_df = pd.read_csv(review_path)

print("Total candidates:", len(review_df))
print("\nReview decisions:")
print(review_df["review_status"].value_counts())

pending = review_df[review_df["review_status"] == "pending"]
print("\nRemaining pending reviews:", len(pending))

accepted = review_df[review_df["review_status"] == "accepted"]

print("Accepted products:", len(accepted))
print("Descriptions exceeding the limit:",
      (accepted["product_description"].str.len() >= 180).sum())
print("Missing descriptions:",
      accepted["product_description"].isna().sum())

Total candidates: 79

Review decisions:
review_status
rejected    43
accepted    36
Name: count, dtype: int64

Remaining pending reviews: 0
Accepted products: 36
Descriptions exceeding the limit: 0
Missing descriptions: 0


In [ ]:
# Export manually accepted products to the required CSV format
review_df = pd.read_csv(review_path)

final_columns = [
    "company_name",
    "stock_name",
    "filing_time",
    "new_product",
    "product_description"
]

# Ensure every candidate has been reviewed
if not review_df["review_status"].isin(["accepted", "rejected"]).all():
    raise ValueError("Some product candidates have not been reviewed.")

# Keep only manually accepted products
final_df = review_df.loc[
    review_df["review_status"] == "accepted",
    final_columns
].copy()

# Validate the final dataset
if final_df.isna().any().any():
    raise ValueError("The final dataset contains missing values.")

if (final_df["product_description"].str.len() >= 180).any():
    raise ValueError("A description exceeds the 179-character limit.")

# Save the final five-column CSV
final_path = os.path.join(drive_output_dir, "sec_new_products_final.csv")
final_df.to_csv(final_path, index=False)

print("Final accepted products:", len(final_df))
print("Final CSV saved to:", final_path)

display(final_df)

Final accepted products: 36
Final CSV saved to: /content/drive/MyDrive/college/Grad/AI/Assignment 2/sec_new_products_final.csv


,company_name,stock_name,filing_time,new_product,product_description
0,Apple Inc.,AAPL,2025-10-30,AirPods Pro 3,Apple announced the launch of AirPods Pro 3.
1,Apple Inc.,AAPL,2025-10-30,MacBook Pro,Apple referenced a recently announced MacBook ...
2,Apple Inc.,AAPL,2025-10-30,iPhone 17,Apple launched iPhone 17 in September as part ...
3,Apple Inc.,AAPL,2025-10-30,iPhone 17 Pro,Apple launched iPhone 17 Pro in September as p...
4,Apple Inc.,AAPL,2025-10-30,iPhone Air,Apple launched iPhone Air in September as part...
6,Apple Inc.,AAPL,2026-04-30,MacBook Neo,"Apple launched MacBook Neo, describing strong ..."
8,Apple Inc.,AAPL,2026-07-30,Siri AI,Apple introduced the all-new Siri AI at WWDC26...
10,ADVANCED MICRO DEVICES INC,AMD,2026-02-03,AMD Instinct MI440X GPU,GPU announced for enterprise AI applications.
11,ADVANCED MICRO DEVICES INC,AMD,2026-02-03,AMD Ryzen AI 400 and PRO 400 Series platforms,New Ryzen AI 400 and PRO 400 Series computing ...
12,ADVANCED MICRO DEVICES INC,AMD,2026-02-03,AMD Ryzen AI Embedded Processor portfolio,New processor portfolio designed for AI applic...


In [ ]:
# Summarize the final extraction and manual review
print("Filings analyzed:", len(company_tickers) * 10)
print("Product candidates:", len(review_df))
print("Accepted products:", len(final_df))
print("Rejected candidates:", (review_df["review_status"] == "rejected").sum())
print("Companies with accepted products:", final_df["stock_name"].nunique())

display(
    final_df.groupby("stock_name")
    .size()
    .reset_index(name="accepted_products")
)

Filings analyzed: 100
Product candidates: 79
Accepted products: 36
Rejected candidates: 43
Companies with accepted products: 5


,stock_name,accepted_products
0,AAPL,7
1,AMD,12
2,AMZN,7
3,META,2
4,NVDA,8
